<a href="https://colab.research.google.com/github/aravinth-xuno/fraud-detection-notebook/blob/main/Pipeline_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
test_df = pd.read_csv("/content/drive/Shareddrives/transaction-training-set/dataset/test_dataset.csv")

In [ ]:
test_df.shape

In [ ]:
test_df['INITIATED_AT'] = pd.to_datetime(test_df['INITIATED_AT'], format="ISO8601", utc=True)

test_df = test_df.sort_values(by=['USER_ID', 'INITIATED_AT']).reset_index(drop=True)

grouped = test_df.groupby('USER_ID')

test_df['cnt_1h'] = grouped.rolling('1h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

test_df['last_1hour_count'] = test_df['cnt_1h']

test_df['cnt_total_25h'] = grouped.rolling('25h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

test_df['last_24hour_count'] = grouped.rolling('24h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

test_df['last_24hour_count_shifted'] = test_df['cnt_total_25h'] - test_df['cnt_1h']

test_df['last_30days_count'] = grouped.rolling('30D', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

test_df['cnt_total_31d'] = grouped.rolling('31D', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

test_df['last_30days_count_shifted'] = test_df['cnt_total_31d'] - test_df['last_24hour_count']

test_df.drop(columns=['cnt_total_25h', 'cnt_total_31d'], inplace=True)

test_df['ratio_1hr_24hr'] = (test_df['last_1hour_count']+1) / (test_df['last_24hour_count_shifted']+1)
test_df['log_ratio_1hr_24hr'] = np.log1p(test_df['ratio_1hr_24hr'])


test_df['ratio_24hr_30day'] = (test_df['last_24hour_count']+1) / (test_df['last_30days_count_shifted']+1)
test_df['log_ratio_24hr_30day'] = np.log1p(test_df['ratio_24hr_30day'])

In [ ]:
features = [
    "last_1hour_count",
    "last_24hour_count",
    "last_30days_count",
    "ratio_1hr_24hr",
    "ratio_24hr_30day"
]

In [ ]:
import pickle
with open("/content/drive/Shareddrives/transaction-training-set/Models/v0.01_high_velocity_model", "rb") as model_file:
    trained_model = pickle.load(model_file)
    print("Model Loaded Successfully")

In [ ]:
test_df["predict"] = trained_model.predict(test_df[features])

In [ ]:
test_df["predict"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score,
                             precision_score, recall_score, f1_score,
                             average_precision_score)

def plot_confusion_matrix(y_actual_raw, y_pred_raw):
    """
    Corrected evaluation function for Isolation Forest.
    Isolation Forest outputs: 1 (Normal), -1 (Anomaly).
    We map them to: 0 (NotFraud), 1 (Fraud).
    """

    # 1. Map labels WITHOUT using inplace (avoiding data corruption)
    # Logic: If it's -1 (anomaly), it's 1 (Fraud). If it's 1 (normal), it's 0 (NotFraud).
    y_actual = (y_actual_raw == -1).astype(int)
    y_pred = (y_pred_raw == -1).astype(int)

    # 2. Print distribution check
    print("--- Label Distribution ---")
    print(f"Actual:\n{y_actual.value_counts()}")
    print(f"Predicted:\n{y_pred.value_counts()}\n")

    # 3. Generate Classification Report
    # 'Fraud' must be the positive class (1)
    report = classification_report(y_actual, y_pred, target_names=['NotFraud', 'Fraud'])
    print("--- Classification Report ---")
    print(report)

    # 4. Standard Metrics (Fraud-centric)
    print(f"Accuracy:     {accuracy_score(y_actual, y_pred):.4f}")
    print(f"Precision:    {precision_score(y_actual, y_pred):.4f}")
    print(f"Recall:       {recall_score(y_actual, y_pred):.4f}")
    print(f"F1 Score:     {f1_score(y_actual, y_pred):.4f}")
    print(f"PR-AUC Score: {average_precision_score(y_actual, y_pred):.4f}")

    # 5. Correct Confusion Matrix Display
    # Scikit-learn puts 0 at [0,0] and 1 at [1,1].
    # Label them in that specific order: ['NotFraud', 'Fraud']
    cm = confusion_matrix(y_actual, y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NotFraud', 'Fraud'])

    fig, ax = plt.subplots(figsize=(8, 6))
    display.plot(ax=ax, cmap='viridis')
    plt.title("Fraud Detection Confusion Matrix")
    plt.show()

In [ ]:
plot_confusion_matrix(test_df["actual"], test_df["predict"])